# Phase 4 ΓÇö Feature Engineering

This notebook compares the existing baseline pipeline with candidate engineered features on the Ames Housing dataset. It does not repeat EDA, modify `src/train_model.py`, save a model bundle, or change the Streamlit app.

Both models use the same 80/20 holdout split (`random_state=42`). Preprocessing is fitted within each training pipeline to avoid validation leakage.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.20

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents]
             if (p / "data/raw/ames_housing/train.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook inside the project repository.")
RAW_PATH = ROOT / "data/raw/ames_housing/train.csv"
CLEAN_PATH = ROOT / "data/processed/ames_housing_clean.csv"
print("Project root:", ROOT)
print("Cleaned dataset exists:", CLEAN_PATH.exists())

Project root: C:\Users\Aditya Khurana\Desktop\Idc409-Group18-House-Prediction
Cleaned dataset exists: True


## 1. Load and validate cleaned data

Preserve literal `"None"` as a valid category. Convert blank fields to missing values, and restore numeric column types using the raw dataset's schema.

In [2]:
raw = pd.read_csv(RAW_PATH)
if not CLEAN_PATH.exists():
    raise FileNotFoundError(f"Complete Phase 2 first; missing {CLEAN_PATH}")

df = pd.read_csv(CLEAN_PATH, keep_default_na=False)
df = df.replace(r"^\s*$", np.nan, regex=True)
for col in raw.columns:
    if col in df.columns and pd.api.types.is_numeric_dtype(raw[col]):
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "SalePrice" not in df.columns:
    raise ValueError("SalePrice target column is missing.")
df = df.dropna(subset=["SalePrice"]).copy()
print("Shape:", df.shape)
print("Literal PoolQC 'None' count:", int(df["PoolQC"].eq("None").sum()))
print("Remaining missing cells:", int(df.isna().sum().sum()))

Shape: (1460, 81)
Literal PoolQC 'None' count: 1453
Remaining missing cells: 82


## 2. Recreate the current baseline pipeline

This matches the preprocessing and model settings in `src/train_model.py`. This notebook evaluates holdout performance only; it does not refit or save the deployment model.

In [7]:
def build_pipeline(X):
    numeric = X.select_dtypes(include=["number"]).columns.tolist()
    categorical = X.select_dtypes(
    include=["object", "str", "category", "bool"]
).columns.tolist()
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric),
        ("categorical", categorical_pipeline, categorical),
    ])
    model = HistGradientBoostingRegressor(
        max_iter=200, learning_rate=0.05, max_leaf_nodes=20,
        l2_regularization=1.0, random_state=RANDOM_STATE,
    )
    return Pipeline([("preprocessor", preprocessor), ("model", model)])

def split_features(data):
    X = data.drop(columns=["SalePrice", "Id"], errors="ignore").copy()
    y = data["SalePrice"].copy()
    return train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

def evaluate_model(name, X_train, X_valid, y_train, y_valid):
    pipeline = build_pipeline(X_train)
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_valid)
    return {
        "Model": name,
        "RMSE ($)": float(np.sqrt(mean_squared_error(y_valid, pred))),
        "MAE ($)": float(mean_absolute_error(y_valid, pred)),
        "R┬▓": float(r2_score(y_valid, pred)),
        "Pipeline": pipeline,
    }

X_train_base, X_valid_base, y_train, y_valid = split_features(df)
baseline = evaluate_model("Baseline", X_train_base, X_valid_base, y_train, y_valid)
pd.DataFrame([{k:v for k,v in baseline.items() if k != "Pipeline"}])

,Model,RMSE ($),MAE ($),R┬▓
0,Baseline,28703.825162,16398.627019,0.892585


## 3. Add candidate housing features

- `TotalSF`: basement + first-floor + second-floor area.
- `TotalBathrooms`: full bathrooms + half bathrooms weighted by 0.5, including basement bathrooms.
- `TotalPorchSF`: sum of porch and deck areas.
- `HouseAge`: sale year minus construction year.
- `RemodAge`: sale year minus remodel year.

Original features are retained so this comparison tests whether adding these candidates helps.

In [8]:
def add_housing_features(data):
    engineered = data.copy()

    def num(name):
        if name in engineered.columns:
            return pd.to_numeric(engineered[name], errors="coerce").fillna(0)
        return pd.Series(0.0, index=engineered.index)

    engineered["TotalSF"] = num("TotalBsmtSF") + num("1stFlrSF") + num("2ndFlrSF")
    engineered["TotalBathrooms"] = (
        num("FullBath") + 0.5*num("HalfBath")
        + num("BsmtFullBath") + 0.5*num("BsmtHalfBath")
    )
    porch_cols = ["OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch", "WoodDeckSF"]
    engineered["TotalPorchSF"] = sum((num(c) for c in porch_cols),
                                     start=pd.Series(0.0, index=engineered.index))
    engineered["HouseAge"] = (num("YrSold") - num("YearBuilt")).clip(lower=0)
    engineered["RemodAge"] = (num("YrSold") - num("YearRemodAdd")).clip(lower=0)
    return engineered

new_features = ["TotalSF", "TotalBathrooms", "TotalPorchSF", "HouseAge", "RemodAge"]
df_engineered = add_housing_features(df)
display(df_engineered[new_features].describe().T)
print("Rows preserved:", len(df_engineered) == len(df))

,count,mean,std,min,25%,50%,75%,max
TotalSF,1460.0,2567.048630,821.714421,334.0,2009.5,2474.0,3004.0,11752.0
TotalBathrooms,1460.0,2.210616,0.785399,1.0,2.0,2.0,2.5,6.0
TotalPorchSF,1460.0,181.329452,156.656097,0.0,45.0,164.0,266.0,1027.0
HouseAge,1460.0,36.547945,30.250152,0.0,8.0,35.0,54.0,136.0
RemodAge,1460.0,22.950685,20.639875,0.0,4.0,14.0,41.0,60.0


Rows preserved: True


## 4. Compare on the same holdout split

The target split must be identical between baseline and engineered versions. Imputers and encoders are fitted only on each training split.

In [9]:
X_train_eng, X_valid_eng, y_train_eng, y_valid_eng = split_features(df_engineered)
assert y_train.equals(y_train_eng) and y_valid.equals(y_valid_eng), "Target split differs."

engineered_result = evaluate_model(
    "Engineered features", X_train_eng, X_valid_eng, y_train_eng, y_valid_eng
)
comparison = pd.DataFrame([
    {k:v for k,v in baseline.items() if k != "Pipeline"},
    {k:v for k,v in engineered_result.items() if k != "Pipeline"},
]).set_index("Model")
display(comparison.round({"RMSE ($)": 2, "MAE ($)": 2, "R┬▓": 4}))

rmse_change_pct = 100 * (baseline["RMSE ($)"] - engineered_result["RMSE ($)"]) / baseline["RMSE ($)"]
mae_change_pct = 100 * (baseline["MAE ($)"] - engineered_result["MAE ($)"]) / baseline["MAE ($)"]
print(f"RMSE improvement: {rmse_change_pct:.2f}% (positive means lower RMSE)")
print(f"MAE improvement:  {mae_change_pct:.2f}% (positive means lower MAE)")

,RMSE ($),MAE ($),R┬▓
Model,,,
Baseline,28703.83,16398.63,0.8926
Engineered features,28964.10,16729.37,0.8906


RMSE improvement: -0.91% (positive means lower RMSE)
MAE improvement:  -2.02% (positive means lower MAE)


## 5. Validity checks and decision

Do not assume added features improve the model. Review the holdout metrics and feature definitions. This is a single holdout comparison, not proof of generalization. No model bundle or application files are modified by this notebook.

In [10]:
display(df_engineered[new_features].agg(["min", "max"]).T)
assert not df_engineered[new_features].isna().any().any()
assert (df_engineered[new_features] >= 0).all().all()

if rmse_change_pct > 0 and mae_change_pct > 0:
    print("Initial result: both RMSE and MAE improved on this holdout.")
elif rmse_change_pct < 0 and mae_change_pct < 0:
    print("Initial result: both RMSE and MAE worsened on this holdout.")
else:
    print("Mixed result: inspect both metrics before deciding.")
print("Phase 4 comparison complete. Training script and app remain unchanged.")

,min,max
TotalSF,334.0,11752.0
TotalBathrooms,1.0,6.0
TotalPorchSF,0.0,1027.0
HouseAge,0.0,136.0
RemodAge,0.0,60.0


Initial result: both RMSE and MAE worsened on this holdout.
Phase 4 comparison complete. Training script and app remain unchanged.
